## Step 1: Set Up Google Colab

Google Colab was used as the Python environment for the RFID data preparation. Google Drive was connected so the simulated RFID baggage tracking file could be accessed during the preparation process.

In [1]:
# Step 1: Set up Google Colab

# Load the libraries

import pandas as pd

# Connect Google Drive

from google.colab import drive
drive.mount('/content/drive')

print("Google Drive is connected.")

Mounted at /content/drive
Google Drive is connected.


## Step 2: Connect the RFID Data

The simulated RFID baggage tracking file was connected from the Praxis project folder. The corrected dataset created in the simulation notebook will be used for the remaining RFID data preparation.

In [2]:
# Step 2: Connect the RFID data

# Set the main project folder

main_folder = "/content/drive/MyDrive/Praxis_Coding"

# Set the RFID folder

rfid_folder = main_folder + "/01_Simulated_RFID_Data"

# Set the RFID file

rfid_file = rfid_folder + "/01_Simulated_RFID_Data.csv"

# Load the RFID data

rfid_data = pd.read_csv(rfid_file)

print("The RFID data is connected.")

The RFID data is connected.


## Step 3: Review the RFID Data

The simulated RFID baggage tracking data was reviewed to confirm the number of scan records, baggage journeys, route assignments, and journey status before the data preparation process continued.

In [3]:
# Step 3: Review the RFID data

# Check the dataset size

print("Total RFID records:", len(rfid_data))

print(
    "Total baggage journeys:",
    rfid_data["bag_id"].nunique()
)

print(
    "Unique scan IDs:",
    rfid_data["scan_id"].nunique()
)

# Review the baggage journeys

bag_check = rfid_data.drop_duplicates("bag_id")

print("\nRoute counts:")
print(
    bag_check["route_type"].value_counts()
)

print("\nJourney status:")
print(
    bag_check["journey_status"].value_counts()
)

# Check the denial of service records

print(
    "\nDoS scans:",
    (rfid_data["disruption_window"] == 1).sum()
)

Total RFID records: 31575
Total baggage journeys: 5000
Unique scan IDs: 31575

Route counts:
route_type
Direct      3500
Transfer    1500
Name: count, dtype: int64

Journey status:
journey_status
Normal      3500
Abnormal    1500
Name: count, dtype: int64

DoS scans: 75


## Step 4: Prepare the RFID Data

The RFID scan records were prepared by comparing the expected and recorded reader, location, timing, and sequence values. The prepared fields provide the information needed for the remaining Zero Trust and cybersecurity analysis.

In [4]:
# Step 4: Prepare the RFID data

# Convert the timestamps

rfid_data["timestamp"] = pd.to_datetime(
    rfid_data["timestamp"]
)

rfid_data["expected_timestamp"] = pd.to_datetime(
    rfid_data["expected_timestamp"]
)

# Compare the reader values

rfid_data["reader_match"] = (
    rfid_data["reader_id"] ==
    rfid_data["expected_reader_id"]
).astype(int)

# Compare the location values

rfid_data["location_match"] = (
    rfid_data["reader_location"] ==
    rfid_data["expected_reader_location"]
).astype(int)

# Compare the sequence values

rfid_data["sequence_match"] = (
    rfid_data["actual_sequence"] ==
    rfid_data["expected_sequence"]
).astype(int)

# Calculate the timing difference

rfid_data["time_difference_minutes"] = (
    (
        rfid_data["timestamp"] -
        rfid_data["expected_timestamp"]
    ).dt.total_seconds() / 60
).abs()

# Identify delayed scans

rfid_data["delayed_indicator"] = (
    rfid_data["time_difference_minutes"] > 15
).astype(int)

# Calculate the time between scans

rfid_data = rfid_data.sort_values(
    ["bag_id", "timestamp"]
).reset_index(drop=True)

rfid_data["scan_gap_minutes"] = (
    rfid_data.groupby("bag_id")["timestamp"]
    .diff()
    .dt.total_seconds() / 60
)

rfid_data["scan_gap_minutes"] = (
    rfid_data["scan_gap_minutes"].fillna(0)
)

# Create the scan label

rfid_data["final_label"] = "Normal"

rfid_data.loc[
    rfid_data["scan_status"] != "Normal",
    "final_label"
] = "Abnormal"

# Create the spoofing label

rfid_data["spoofing_label"] = 0

rfid_data.loc[
    rfid_data["scan_status"] == "Spoofed Reader",
    "spoofing_label"
] = 1

# Check the prepared fields

print(
    "Reader mismatches:",
    (rfid_data["reader_match"] == 0).sum()
)

print(
    "Location mismatches:",
    (rfid_data["location_match"] == 0).sum()
)

print(
    "Sequence mismatches:",
    (rfid_data["sequence_match"] == 0).sum()
)

print(
    "Delayed scans:",
    rfid_data["delayed_indicator"].sum()
)

print(
    "Spoofed scans:",
    rfid_data["spoofing_label"].sum()
)

Reader mismatches: 375
Location mismatches: 225
Sequence mismatches: 225
Delayed scans: 225
Spoofed scans: 375


## Step 5: Validate and Save the Prepared Data

The prepared RFID dataset was reviewed before being saved for the Zero Trust, machine learning, and denial of service analysis.

In [5]:
# Step 5: Validate and save the prepared data

# Check the prepared dataset

print("Total RFID records:", len(rfid_data))

print(
    "Total baggage journeys:",
    rfid_data["bag_id"].nunique()
)

print(
    "Unique scan IDs:",
    rfid_data["scan_id"].nunique()
)

print("\nFinal scan labels:")
print(
    rfid_data["final_label"].value_counts()
)

print(
    "\nSpoofed scans:",
    rfid_data["spoofing_label"].sum()
)

print(
    "DoS scans:",
    (rfid_data["disruption_window"] == 1).sum()
)

# Set the prepared data folder

prepared_folder = main_folder + "/06_Prepared_Data"

# Set the prepared RFID file

prepared_file = (
    prepared_folder +
    "/02_Prepared_RFID_Data.csv"
)

# Save the prepared data

rfid_data.to_csv(
    prepared_file,
    index=False
)

print("\nPrepared RFID data saved:")
print(prepared_file)

Total RFID records: 31575
Total baggage journeys: 5000
Unique scan IDs: 31575

Final scan labels:
final_label
Normal      30225
Abnormal     1350
Name: count, dtype: int64

Spoofed scans: 375
DoS scans: 75

Prepared RFID data saved:
/content/drive/MyDrive/Praxis_Coding/06_Prepared_Data/02_Prepared_RFID_Data.csv


In [6]:
# Check the missing scan journeys

print(
    "\nMissing scan journeys:",
    rfid_data[
        rfid_data["journey_abnormal_type"] == "Missing Scan"
    ]["bag_id"].nunique()
)


Missing scan journeys: 150
